# Generazione embeddings

Questo notebook legge i testi gia' preparati per il retrieval semantico e genera un vettore embedding per ogni record.

Input:
- `data/processed/embedding_corpus.jsonl`
- `data/processed/embedding_queries.jsonl`

Output:
- `data/processed/corpus_embeddings.jsonl`
- `data/processed/query_embeddings.jsonl`

Non vengono creati vector database o API FastAPI: questo step salva solo i vettori su file.

## 1. Configurazione

Il modello locale, il batch size e i percorsi di output sono configurabili qui. Non servono API key.

In [ ]:
import json
import os
from pathlib import Path
from typing import Any


current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
DATA_DIR = PROJECT_ROOT / "data" / "processed"

CORPUS_INPUT_PATH = DATA_DIR / "embedding_corpus.jsonl"
QUERIES_INPUT_PATH = DATA_DIR / "embedding_queries.jsonl"

def configured_path(env_var: str, default_path: Path) -> Path:
    value = os.getenv(env_var)
    path = Path(value) if value else default_path
    return path if path.is_absolute() else PROJECT_ROOT / path


# Configurazione principale
MODEL_NAME = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
BATCH_SIZE = int(os.getenv("EMBEDDING_BATCH_SIZE", "64"))
if BATCH_SIZE <= 0:
    raise ValueError("EMBEDDING_BATCH_SIZE deve essere maggiore di zero")

# Percorsi output: si possono cambiare qui oppure via env var.
CORPUS_OUTPUT_PATH = configured_path("CORPUS_EMBEDDINGS_PATH", DATA_DIR / "corpus_embeddings.jsonl")
QUERIES_OUTPUT_PATH = configured_path("QUERY_EMBEDDINGS_PATH", DATA_DIR / "query_embeddings.jsonl")

CORPUS_INPUT_PATH, QUERIES_INPUT_PATH, CORPUS_OUTPUT_PATH, QUERIES_OUTPUT_PATH, MODEL_NAME, BATCH_SIZE

## 2. Funzioni semplici per JSONL e batch

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON non valido in {path} alla riga {line_number}: {exc}") from exc
    return records


def write_jsonl_atomic(records: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")

    with temp_path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    temp_path.replace(path)


def make_batches(items: list[Any], batch_size: int) -> list[list[Any]]:
    return [items[start:start + batch_size] for start in range(0, len(items), batch_size)]

## 3. Lettura dati e controlli sugli input

In [ ]:
corpus_records = read_jsonl(CORPUS_INPUT_PATH)
query_records = read_jsonl(QUERIES_INPUT_PATH)

print(f"Corpus records: {len(corpus_records):,}")
print(f"Query records:  {len(query_records):,}")

In [ ]:
REQUIRED_FIELDS = {"record_id", "source_image", "embedding_text", "metadata"}


def validate_input_records(records: list[dict[str, Any]], dataset_name: str) -> None:
    missing_fields = []
    empty_texts = []

    for index, record in enumerate(records):
        missing = REQUIRED_FIELDS - set(record)
        if missing:
            missing_fields.append((index, sorted(missing)))

        text = record.get("embedding_text", "")
        if not isinstance(text, str) or not text.strip():
            empty_texts.append(record.get("record_id"))

    print(f"{dataset_name}: record controllati = {len(records):,}")
    print(f"{dataset_name}: testi vuoti = {len(empty_texts):,}")
    print(f"{dataset_name}: record con campi mancanti = {len(missing_fields):,}")

    if missing_fields[:5]:
        print("Esempi campi mancanti:", missing_fields[:5])
    if empty_texts[:10]:
        print("Esempi record_id con testo vuoto:", empty_texts[:10])

    if missing_fields or empty_texts:
        raise ValueError(f"Input non valido per {dataset_name}. Correggere prima di generare embeddings.")


validate_input_records(corpus_records, "corpus")
validate_input_records(query_records, "queries")

## 4. Modello embedding locale

Questo notebook usa `sentence-transformers`, quindi non richiede API key.

Modello default: `sentence-transformers/all-MiniLM-L6-v2`.

La prima esecuzione puo' scaricare il modello da Hugging Face; dopo viene riusata la cache locale.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError(
        "Installa sentence-transformers prima di eseguire questo notebook: "
        "pip install sentence-transformers"
    ) from exc


embedding_model = SentenceTransformer(MODEL_NAME)


def embed_texts(texts: list[str]) -> list[list[float]]:
    vectors = embedding_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return vectors.tolist()

## 5. Generazione embeddings

In [ ]:
def build_embedding_records(records: list[dict[str, Any]], dataset_name: str) -> list[dict[str, Any]]:
    output_records = []
    expected_dimension = None
    batches = make_batches(records, BATCH_SIZE)

    for batch_number, batch in enumerate(batches, start=1):
        texts = [record["embedding_text"].strip() for record in batch]
        vectors = embed_texts(texts)

        if len(vectors) != len(batch):
            raise ValueError(f"{dataset_name}: il batch {batch_number} ha prodotto un numero inatteso di vettori")

        for input_record, vector in zip(batch, vectors):
            if expected_dimension is None:
                expected_dimension = len(vector)
            elif len(vector) != expected_dimension:
                raise ValueError(
                    f"{dataset_name}: dimensione vettore non coerente per record_id={input_record['record_id']}"
                )

            output_records.append({
                "record_id": input_record["record_id"],
                "source_image": input_record["source_image"],
                "metadata": input_record["metadata"],
                "embedding_model": MODEL_NAME,
                "embedding_dimension": len(vector),
                "embedding": vector,
            })

        print(f"{dataset_name}: batch {batch_number}/{len(batches)} completato")

    print(f"{dataset_name}: record processati = {len(output_records):,}")
    print(f"{dataset_name}: dimensione vettori = {expected_dimension}")
    return output_records


corpus_embedding_records = build_embedding_records(corpus_records, "corpus")
query_embedding_records = build_embedding_records(query_records, "queries")

## 6. Controlli finali

In [ ]:
def validate_embedding_output(input_records: list[dict[str, Any]], output_records: list[dict[str, Any]], dataset_name: str) -> None:
    if len(input_records) != len(output_records):
        raise ValueError(f"{dataset_name}: numero record input/output diverso")

    input_ids = [record["record_id"] for record in input_records]
    output_ids = [record["record_id"] for record in output_records]
    if input_ids != output_ids:
        raise ValueError(f"{dataset_name}: record_id non preservati nello stesso ordine")

    dimensions = {record["embedding_dimension"] for record in output_records}
    if len(dimensions) != 1:
        raise ValueError(f"{dataset_name}: dimensioni vettori multiple: {sorted(dimensions)}")

    empty_vectors = [record["record_id"] for record in output_records if not record["embedding"]]
    if empty_vectors:
        raise ValueError(f"{dataset_name}: vettori vuoti trovati: {empty_vectors[:10]}")

    print(f"{dataset_name}: OK")
    print(f"  record = {len(output_records):,}")
    print(f"  dimensione = {next(iter(dimensions))}")
    print(f"  modello = {MODEL_NAME}")


validate_embedding_output(corpus_records, corpus_embedding_records, "corpus")
validate_embedding_output(query_records, query_embedding_records, "queries")

## 7. Salvataggio output

Il salvataggio e' atomico: prima scrive un file `.tmp`, poi lo sostituisce al file finale. In questo modo una riesecuzione non accoda record e non lascia output duplicati.

In [ ]:
write_jsonl_atomic(corpus_embedding_records, CORPUS_OUTPUT_PATH)
write_jsonl_atomic(query_embedding_records, QUERIES_OUTPUT_PATH)

print(f"Salvato corpus:  {CORPUS_OUTPUT_PATH}")
print(f"Salvate queries: {QUERIES_OUTPUT_PATH}")

## 8. Lettura di verifica

In [ ]:
saved_corpus = read_jsonl(CORPUS_OUTPUT_PATH)
saved_queries = read_jsonl(QUERIES_OUTPUT_PATH)

print(f"Corpus salvato: {len(saved_corpus):,} record")
print(f"Queries salvate: {len(saved_queries):,} record")
print("Esempio campi output:", list(saved_corpus[0].keys()))
print("Dimensione esempio:", saved_corpus[0]["embedding_dimension"])